# Phase 3 -- Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import os
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('credit_card_fraud_dataset.csv')
print(f"Loaded: {df.shape}")

Loaded: (500, 17)


### 3.1 -- Handle Categorical Features

In [2]:
# encoding day_of_week with a fixed mapping so the order makes sense
day_mapping = {'Mon': 0, 'Tue': 1, 'Wed': 2, 'Thu': 3, 'Fri': 4, 'Sat': 5, 'Sun': 6}
df['day_of_week'] = df['day_of_week'].map(day_mapping)

print("day_of_week encoded:")
print(df['day_of_week'].value_counts().sort_index())

day_of_week encoded:
day_of_week
0    79
1    65
2    73
3    70
4    93
5    74
6    46
Name: count, dtype: int64


In [3]:
# label encoding for merchant_category
le_merchant = LabelEncoder()
df['merchant_category'] = le_merchant.fit_transform(df['merchant_category'])

print("merchant_category encoded:")
for i, cls in enumerate(le_merchant.classes_):
    print(f"  {cls} -> {i}")

merchant_category encoded:
  cash_advance -> 0
  electronics -> 1
  entertainment -> 2
  fuel -> 3
  grocery -> 4
  healthcare -> 5
  jewelry -> 6
  online_shopping -> 7
  restaurant -> 8
  retail_clothing -> 9
  travel -> 10
  utilities -> 11


In [4]:
# verify no NaN after encoding
print(f"NaN values after encoding: {df.isnull().sum().sum()}")

NaN values after encoding: 0


### 3.2 -- Separate Features (X) and Target (y)

In [5]:
# drop transaction_id -- it is just a row identifier, not a real feature
df.drop('transaction_id', axis=1, inplace=True)

X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

print(f"Features (X): {X.shape}")
print(f"Target  (y): {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

Features (X): (500, 15)
Target  (y): (500,)

Feature columns: ['transaction_amount_inr', 'transaction_hour', 'day_of_week', 'merchant_category', 'is_high_risk_merchant', 'card_present', 'location_match', 'distance_from_home_km', 'prev_txn_gap_mins', 'num_txn_last_24h', 'num_declined_last_7days', 'cvv_mismatch', 'international_transaction', 'customer_age_years', 'account_age_days']


### 3.3 -- Train/Test Split

Splitting before scaling is critical. We must never fit the scaler on test data.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts()}")
print(f"y_test distribution:\n{y_test.value_counts()}")

X_train: (400, 15)
X_test:  (100, 15)
y_train distribution:
is_fraud
0    380
1     20
Name: count, dtype: int64
y_test distribution:
is_fraud
0    95
1     5
Name: count, dtype: int64


### 3.4 -- Feature Scaling

Fit on training data only, then transform both train and test.

In [7]:
scaler = StandardScaler()

# fit ONLY on training data
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

# transform test data using the same scaler
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Scaling done. Quick check on scaled training data:")
X_train_scaled.describe().round(2)

Scaling done. Quick check on scaled training data:


,transaction_amount_inr,transaction_hour,day_of_week,merchant_category,is_high_risk_merchant,card_present,location_match,distance_from_home_km,prev_txn_gap_mins,num_txn_last_24h,num_declined_last_7days,cvv_mismatch,international_transaction,customer_age_years,account_age_days
count,400.00,400.00,400.00,400.00,400.00,400.00,400.00,400.00,400.00,400.00,400.00,400.00,400.00,400.00,400.00
mean,0.00,-0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,-0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-0.57,-3.09,-1.50,-1.96,-0.47,-1.88,-2.71,-0.24,-1.68,-0.79,-0.43,-0.25,-0.33,-1.92,-1.67
25%,-0.46,-0.72,-0.98,-0.60,-0.47,0.53,0.37,-0.22,-0.88,-0.40,-0.43,-0.25,-0.33,-0.90,-0.81
50%,-0.37,-0.01,0.05,-0.26,-0.47,0.53,0.37,-0.19,0.03,-0.01,-0.43,-0.25,-0.33,0.01,-0.08
75%,-0.09,0.70,0.57,0.76,-0.47,0.53,0.37,-0.17,0.89,0.38,-0.43,-0.25,-0.33,0.86,0.92
max,6.38,2.35,1.61,1.78,2.12,0.53,0.37,9.72,1.72,6.24,4.49,4.05,3.00,1.71,1.78


### 3.5 -- Save Preprocessing Artifacts

In [8]:
os.makedirs('models', exist_ok=True)

# save scaler
joblib.dump(scaler, 'models/scaler.pkl')
print("Saved: models/scaler.pkl")

# save label encoder for merchant_category
# also saving the day_mapping so the dashboard can use it
encoders = {
    'merchant_category_encoder': le_merchant,
    'day_of_week_mapping': day_mapping
}
joblib.dump(encoders, 'models/label_encoder.pkl')
print("Saved: models/label_encoder.pkl")

Saved: models/scaler.pkl
Saved: models/label_encoder.pkl


In [9]:
# quick verify -- load them back and check
loaded_scaler = joblib.load('models/scaler.pkl')
loaded_encoders = joblib.load('models/label_encoder.pkl')

print(f"Scaler type: {type(loaded_scaler).__name__}")
print(f"Encoder keys: {list(loaded_encoders.keys())}")
print(f"Merchant categories: {list(loaded_encoders['merchant_category_encoder'].classes_)}")
print(f"Day mapping: {loaded_encoders['day_of_week_mapping']}")

Scaler type: StandardScaler
Encoder keys: ['merchant_category_encoder', 'day_of_week_mapping']
Merchant categories: ['cash_advance', 'electronics', 'entertainment', 'fuel', 'grocery', 'healthcare', 'jewelry', 'online_shopping', 'restaurant', 'retail_clothing', 'travel', 'utilities']
Day mapping: {'Mon': 0, 'Tue': 1, 'Wed': 2, 'Thu': 3, 'Fri': 4, 'Sat': 5, 'Sun': 6}


---
### Phase 3 Summary

- Encoded `day_of_week` (Mon=0 to Sun=6) and `merchant_category` (LabelEncoder).
- Dropped `transaction_id`.
- Split data 80/20 with stratification.
- Scaled features using StandardScaler fitted on training data only.
- Saved `scaler.pkl` and `label_encoder.pkl` to `models/`.

Ready for Phase 4 (SMOTE).

---